# Inference on One Population Mean when $\sigma^2 $ is Known or when $\sigma^2 $ Unknown (when Sample Size is Large, $n \geq 30$)

## Standard Z Test Procedure

This procedure works for two scenarios:

* When $\sigma^2 $ is known:
In that case the test statistic is given by
$$Z_{CALC}=\dfrac{\bar{X}-\mu }{\left(\dfrac{\sigma }{\sqrt{n}}\right)}$$

* When $\sigma^2 $ is unknown:
In that case the test statistic is given by
$$Z_{CALC}=\dfrac{\bar{X}-\mu }{\left(\dfrac{s }{\sqrt{n}}\right)}$$ 
where $s$ is the sample standard deviation.

The code below provides for the first scenario, however, you can simply substitute the sample standard deviation for the population standard deviation.

In [1]:
import numpy as np
import scipy.stats as stats
import pandas as pd

def Z_test_1sample(sample_mean, sample_std, sample_size, mu_0, alternative, alpha):
    def Z_calc(sample_mean, sample_std, sample_size, mu_0):
        return (sample_mean - mu_0) / (sample_std / np.sqrt(sample_size))
    
    z_value = Z_calc(sample_mean, sample_std, sample_size, mu_0)

#The table value    
    def Z_table(alpha, alternative):
        if alternative == 'two sided':
            p_value = 2 * (1 - stats.norm.cdf(abs(z_value)))
            return stats.norm.ppf(1 - alpha / 2), p_value
        elif alternative == 'greater':
            p_value = 1 - stats.norm.cdf(z_value)
            return stats.norm.ppf(1 - alpha), p_value
        elif alternative == 'less':
            p_value = stats.norm.cdf(z_value)
            return stats.norm.ppf(alpha), p_value
        else:
            raise ValueError('Type either "two sided", "greater" or "less" for the alternative')
            
    critical_value, p_value = Z_table(alpha, alternative)
    
    result = {"Z_calc": z_value, "Z_table": critical_value, "P-value": p_value}
    result_table = pd.DataFrame([result], index=['Values'])
    
    styled_table = table = result_table.style.set_caption('Z Test Results')
    
    return styled_table
    
    #print("Z_calc =", z_value, "  ", "Z_table =", critical_value, "  ", "p-value =", p_value)

In [2]:
# Use case

Z_test_1sample(sample_mean=42196, sample_std=500, sample_size=32, mu_0=42000, alternative='less', alpha=0.05)

,Z_calc,Z_table,P-value
Values,2.217487,-1.644854,0.986705


### Example 1

Data: We download a restaurant tips data (real observations of tips in dollars) from seaborn datasets. (Reference to seaborn datasets: https://www.geeksforgeeks.org/data-science/seaborn-datasets-for-data-science/#1-tips-dataset)

Objective: After observing the data, we hypothesize that the average tip is at least $2.75. We want to test whether our conviction is statistically true. We assume that the population standard deviation is known and is equal to the standard deviation of the tips data.

We want to test: 
$$H_{0}: \quad \mu \leq 2.75 \quad \text{vs} \quad \mu > 2.75 \quad \text{at} \quad \alpha=0.05 $$

In [3]:
import seaborn as sns

tips_data = sns.load_dataset("tips")
tips_data

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [4]:
tips = tips_data['tip']  # Working with only the tip column

print("Population standard deviation is : ", tips.std())
print("Sample mean is : ", tips.mean())
print("Sample size is : ", tips.count())

Population standard deviation is :  1.3836381890011822
Sample mean is :  2.99827868852459
Sample size is :  244


In [5]:
# Test results

Z_test_1sample(sample_mean = tips.mean(), sample_std = tips.std(), sample_size = tips.count(), mu_0 = 2.75, alternative='greater', alpha=0.05)

,Z_calc,Z_table,P-value
Values,2.802927,1.644854,0.002532


**Interpretation**: The sample provides strong statistical evidence that the average tip is greater than $ \$2.75$. The observed average tip is about $ \$2.998$, and the chance of seeing a mean this large (or larger) if the true mean were $2.75 is only about 0.25%.

### Example 2

Data: We simulate 400 independent measurements from the Normal distribution with true mean 50.0 and a standard deviation $\sigma = 5.0$. This is purely to get the simulated data. We blindly assume that we do not know the true standard deviation of this normal distribution.

Objective: We want to test whether the population mean is statistically less than 51.02. 
$$H_{0}: \quad \mu \geq 51.02 \quad \text{vs} \quad \mu < 51.02 $$
at the significance level: $\alpha=0.05 $.

In [6]:
np.random.seed(112)

simu_data = np.random.normal(loc = 50.0, scale = 5.0, size = 400)

print("Sample standard deviation is : ", simu_data.std())
print("Sample mean is : ", simu_data.mean())
print("Sample size is : ", len(simu_data))

Sample standard deviation is :  4.997719812741636
Sample mean is :  50.20079012564526
Sample size is :  400


In [7]:
# Test results

Z_test_1sample(sample_mean = simu_data.mean(), sample_std = simu_data.std(), sample_size = len(simu_data), mu_0 = 51.02, alternative='less', alpha=0.05)

,Z_calc,Z_table,P-value
Values,-3.278335,-1.644854,0.000522


**Interpretation:** We reject the null hypothesis and claim that the simulated data suggests the average is indeed lower than  51.02, and the result is statistically significant.

### Generating only the Z Score, corresponding to a certain $\alpha $

In [8]:
def Z_table(alpha):
    if alpha < 0.5:
        return stats.norm.ppf(alpha)
    if alpha >= 0.5:
        return stats.norm.ppf(1-alpha)

Z_table(alpha=0.1075)

np.float64(-1.2399334778907383)

## Power Calculation for Z - Tests

Power calculation are usually performed in relation to a hypothesis test. We want to know by which probability can we correctly reject the null hypothesis in favour of the alternative hypothesis.

* $H_{0}:\mu = \mu_{0}$ vs $H_{0}:\mu \neq \mu_{0}$ 
$$\text{Power } = \phi \left[-z_{\alpha \mathbin{/}2} + \dfrac{(\mu_{0}- \mu_{a})\sqrt{n}}{\sigma }\right] + \phi \left[-z_{\alpha \mathbin{/}2} + \dfrac{(\mu_{a}- \mu_{0})\sqrt{n}}{\sigma }\right] $$

* $H_{0}:\mu \geq \mu_{0}$ vs $H_{0}:\mu < \mu_{0}$ 
$$\text{Power } = \phi \left[-z_{\alpha } - \dfrac{(\mu_{a}- \mu_{0})\sqrt{n}}{\sigma }\right] $$

* $H_{0}:\mu \leq \mu_{0}$ vs $H_{0}:\mu > \mu_{0}$ 
$$\text{Power } = 1 - \phi \left[z_{\alpha } - \dfrac{(\mu_{a}- \mu_{0})\sqrt{n}}{\sigma }\right] $$

Next, we provide the code for these codes.

In [9]:
import numpy as np
import scipy.stats as stats
import pandas as pd

def Z_power(alpha, mu_0, mu_A, sample_size, population_std, alternative):
        if alternative == 'two sided':
            z_power = stats.norm.cdf(-stats.norm.ppf(1 - alpha / 2) + ((mu_0 - mu_A)*np.sqrt(sample_size))/population_std) + stats.norm.cdf(-stats.norm.ppf(1 - alpha / 2) + ((mu_A - mu_0)*np.sqrt(sample_size))/population_std)
            return z_power
        elif alternative == 'greater':
            z_power = 1 - stats.norm.cdf(stats.norm.ppf(1 - alpha / 2) - ((mu_A - mu_0)*np.sqrt(sample_size))/population_std)
            return  z_power
        elif alternative == 'less':
            z_power = stats.norm.cdf(-stats.norm.ppf(1 - alpha / 2) - ((mu_A - mu_0)*np.sqrt(sample_size))/population_std)
            return  z_power
        else:
            raise ValueError('Type either "two sided", "greater" or "less" for the alternative')

In [10]:
Z_power(0.0027, 16, 16.1, 5, 0.1, 'two sided')

np.float64(0.22246081435606987)

## Sample Size Determination for Z Test

There are three scenarios in which we can determine the right sample size for a statistical z test.

1. For a given power
    * Two Sided Test
$$n=\left[\dfrac{\left(z_{\alpha \mathbin{/}2}+z_{\beta }\right)\sigma }{(\mu_{a}-\mu_{0})}\right]^ {2}$$

    * One Sided Test
$$n=\left[\dfrac{\left(z_{\alpha }+z_{\beta }\right)\sigma }{(\mu_{a}-\mu_{0})}\right]^ {2}$$

2. For a given CI length $L$
    * Two Sided Test
$$n=\left[\dfrac{2 z_{\alpha \mathbin{/}2}\sigma }{L}\right]^ {2}$$

    * One Sided Test
$$n=\left[ \dfrac{2 z_{\alpha }\sigma }{L}\right]^ {2}$$

2. For a given margin of error $E$
    * Two Sided Test
$$n=\left[ \dfrac{z_{\alpha \mathbin{/}2}\sigma }{E}\right]^ {2}$$

    * One Sided Test
$$n=\left[ \dfrac{z_{\alpha }\sigma }{E}\right]^ {2}$$

Now we provide the code for sample size determination for z tests.

#### 1. For a Given Power

In [11]:
import numpy as np
import scipy.stats as stats
import pandas as pd

def sample_size_given_power(alpha,population_std,mu_0,mu_A, power,alternative):
    if alternative == 'two sided':
            z_alpha = stats.norm.ppf(1 - alpha/2)
            beta = 1 - power
            z_beta = stats.norm.ppf(beta)
    
    elif alternative == 'one sided':
            z_alpha = stats.norm.ppf(1 - alpha)
            beta = 1 - power
            z_beta = stats.norm.ppf(1 - beta)
    else:
        raise ValueError('Type either "two sided", or "one sided" for the alternative') 
    
    sample_size_calc = (((z_alpha + z_beta)*population_std)/(mu_A - mu_0))**2
    
    return sample_size_calc

In [12]:
sample_size_given_power(alpha=0.05,population_std=40,mu_0=15,mu_A=30, power=0.90,alternative='one sided')

np.float64(60.89847004919449)

#### 2. & 3. For a given margin of error $E$
We use the same code. However, note that $E=2L$

In [13]:
import numpy as np
import scipy.stats as stats
import pandas as pd

def sample_size_given_error(alpha,population_std, error,alternative):
    if alternative == 'two sided':
            z_alpha = stats.norm.ppf(1 - alpha/2)
    
    elif alternative == 'one sided':
            z_alpha = stats.norm.ppf(1 - alpha)
    else:
        raise ValueError('Type either "two sided", or "one sided" for the alternative') 
        
    sample_size_calc = ((z_alpha *population_std)/error)**2
    
    return sample_size_calc

In [14]:
sample_size_given_error(alpha=0.01,population_std=50, error=5,alternative='two sided')

np.float64(663.4896601021213)

## Confidence Interval on Mean $\mu$ for Z test

The confidence interval for Z test on one population mean works whether we have sample standard deviation or population standard deviation. In the code, we use sample standard deviation but one can easily substitute the value of the population standard deviation in place of sample standard deviation to achieve same result.

* Two Sided CI
$$ \bar{x} - z_{\alpha \mathbin{/} 2}\dfrac{\sigma }{\sqrt{n}} \leq \mu \leq \bar{x} + z_{\alpha \mathbin{/} 2}\dfrac{\sigma }{\sqrt{n}}$$

* Upper One Sided
$$\mu \leq \bar{x} + z_{\alpha}\dfrac{\sigma }{\sqrt{n}}$$

* Lower One Sided
$$ \mu \geq \bar{x} - z_{\alpha }\dfrac{\sigma }{\sqrt{n}}$$

In [17]:
import numpy as np
import scipy.stats as stats
import pandas as pd

def z_confidence_interval_mean(alpha,sample_mean,sample_std,sample_size,alternative):
    if alternative == 'two sided':
        lower_bound = sample_mean - stats.norm.ppf(1 - alpha/2)*((sample_std)/np.sqrt(sample_size))
        upper_bound = sample_mean + stats.norm.ppf(1 - alpha/2)*((sample_std)/np.sqrt(sample_size))
        conf_interval = (lower_bound, upper_bound)
    
    elif alternative == 'greater':
        upper_bound = sample_mean + stats.norm.ppf(1 - alpha)*((sample_std)/np.sqrt(sample_size))
        conf_interval = print("μ ≤ ", upper_bound)
    
    elif alternative == 'less':
        lower_bound = sample_mean - stats.norm.ppf(1 - alpha)*((sample_std)/np.sqrt(sample_size))
        conf_interval = print("μ ≥  ",lower_bound)
    
    else:
        raise ValueError('Type either "two sided", "greater" or "less" for the alternative')
    
    return conf_interval

In [16]:
z_confidence_interval_mean(alpha=0.10,sample_mean=42196,sample_std=500,sample_size=32,alternative='less')

μ ≥   42082.7257746954
